# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading, exploring, and analyzing the "A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya" using the `mlcroissant` library. All entities (record sets, fields, etc.) are referenced by their `@id`, following best practices in Croissant datasets.

### Dataset Source
The dataset is provided via a Croissant schema URL:

```
https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json
```

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata (do not treat as dict/list)
print(f"Dataset name: {dataset.metadata.name}\n")
print(f"Description: {dataset.metadata.description}\n")
print(f"Published: {dataset.metadata.datePublished}\n")

## 2. Data Overview
Review the available record sets, fields, and their `@id`s. This will help navigate the dataset programmatically.

In [ ]:
# List all record sets (referenced by their @id)
print("Available record sets and their @id:")
for recordset in dataset.metadata.record_sets:
    print(f"- name: {recordset.name:<40} @id: {recordset.id}")

# For demonstration, select the main survey record set
# (The exact @id may be different if you explore the metadata structure.)
main_record_set_id = None
for rs in dataset.metadata.record_sets:
    # Here we select based on partial string; replace as needed
    if "survey" in rs.name.lower() or "main" in rs.name.lower() or True:
        main_record_set_id = rs.id
        break
if not main_record_set_id:
    main_record_set_id = dataset.metadata.record_sets[0].id
print(f"\nSelected record set for demonstration: {main_record_set_id}\n")

# List all fields for this record set (referenced by their @id)
print("Fields in the selected record set:")
fields = [field for field in dataset.metadata.fields if field.record_set == main_record_set_id]
for field in fields:
    print(f"- name: {field.name:<35} @id: {field.id}")

## 3. Data Extraction
Load data from the chosen record set into a DataFrame for analysis. Use the record set and field `@id`s from above for programmatic access.

In [ ]:
# Extract data from all record sets and store as DataFrames by their @id
dataframes = {}

for rs in dataset.metadata.record_sets:
    records = list(dataset.records(record_set=rs.id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs.id] = df
        print(f"Loaded DataFrame for record set '{rs.name}' [@id: {rs.id}] with shape: {df.shape}")

# Access the DataFrame for the main record set
main_df = dataframes[main_record_set_id]
print("\nFirst few columns and rows from the main DataFrame:")
print(main_df.columns.tolist())
main_df.head()

## 4. Exploratory Data Analysis (EDA)
Apply basic filtering, normalization, and grouping on numeric records. All columns are referenced by their Field `@id`.

Example: filter by a GAD-7 score field, normalize, and group by education level if available.

In [ ]:
# --- Identify a relevant numeric and group field (by @id) from previous cell's printout. ---
# For demonstration, suppose GAD-7 score column has @id 'gad7_total', education 'level_of_education'.
numeric_field_id = None
group_field_id = None
# Let us try to detect relevant fields:
for c in main_df.columns:
    if 'gad' in c.lower() and 'total' in c.lower():
        numeric_field_id = c
    if 'education' in c.lower():
        group_field_id = c
if not numeric_field_id:
    # Fallback, use any numeric column
    numeric_candidates = main_df.select_dtypes(include='number').columns.tolist()
    numeric_field_id = numeric_candidates[0] if numeric_candidates else main_df.columns[0]
if not group_field_id:
    # Fallback, use any likely categorical
    group_field_id = main_df.columns[1] if len(main_df.columns) > 1 else main_df.columns[0]

print(f"Numeric field selected (by @id): {numeric_field_id}")
print(f"Grouping field selected (by @id): {group_field_id}")

# Filter records where the numeric field is present and greater than a threshold
threshold = 5
filtered_df = main_df[main_df[numeric_field_id] > threshold]
print(f"Filtered records where {numeric_field_id} > {threshold}:")
print(filtered_df[[numeric_field_id, group_field_id]].head())

# Normalize the numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by a key attribute
if group_field_id in filtered_df.columns:
    grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
    print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:")
    print(grouped.head())

## 5. Visualization
Let us visualize the distribution of the numeric field and its relation to the grouping field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution histogram
plt.figure(figsize=(7,4))
sns.histplot(main_df[numeric_field_id].dropna(), bins=15, kde=True, color="dodgerblue")
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# Boxplot by group if grouping field is categorical with reasonable cardinality
if main_df[group_field_id].nunique() < 30:
    plt.figure(figsize=(10,5))
    sns.boxplot(x=main_df[group_field_id], y=main_df[numeric_field_id])
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=30)
    plt.show()

## 6. Conclusion
In this notebook, we loaded the FAIR² mental health survey dataset (referencing all entities by their Croissant `@id`), explored its record sets and fields, extracted survey data, performed basic exploration, and visualized numeric results. Use this workflow as a foundation to further analyze or model the Kilifi dataset in line with reproducible best practices.